
## How to Use the Notebooks

To run these notebooks, follow these steps:


### 1. Install **`gdal`** Package
Installing `gdal` via `pip` may fail. Therefore, we recommend using conda to install it:
```bash
conda install gdal -c conda-forge
```

### 2. Install **`polsartools`** Package

Before running the notebooks, you'll need to install the core **`polsartools`** package. 

**Option A:** Install from PyPI `pip` (stable release)

```bash
pip install polsartools
```

**Option B:** Install from conda (stable release)

```bash
conda install polsartools -c bnarayanarao
```


**Option C:**  Install the latest version from GitHub (if you need recent updates or fixes)

```bash
pip install git+https://github.com/Narayana-Rao/polsartools.git#egg=polsartools
```
> **Note:** If you encounter a `"module not found"` error or require newer features, prefer the **Option C**.

### 3. Verify the Installation

After successfully installing `polsartools`, you can verify it by importing the package:

```bash
import polsartools as pst
```

If this runs without errors, you're ready to explore and run the notebooks.




In [1]:
""" 
import required packages
"""

import os
import matplotlib.pyplot as plt
import numpy as np
from osgeo import gdal
import polsartools as pst
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module='pyproj')


In [2]:
""" 

Function to display images

"""

def plot_images(*filenames, lower_percentile=2, upper_percentile=98, max_cols=4):
    def compute_vmin_vmax(data):
        data_no_nan = data[np.isfinite(data)]
        vmin = np.percentile(data_no_nan, lower_percentile)
        vmax = np.percentile(data_no_nan, upper_percentile)
        return vmin, vmax

    num_images = len(filenames)

    if num_images == 1:
        # Single image case
        data = pst.read_rst(filenames[0])
        data[data==0]=np.nan
        vmin, vmax = compute_vmin_vmax(data)

        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(data, vmin=vmin, vmax=vmax)
        ax.set_title(os.path.basename(filenames[0]))
        fig.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()
    else:
        # Multiple image case
        ncols = min(max_cols, num_images)
        nrows = int(np.ceil(num_images / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
        axes = np.array(axes).reshape(-1)  # Flatten in case it's a 2D grid

        for i, filename in enumerate(filenames):
            data = pst.read_rst(filename)
            data[data==0]=np.nan
            vmin, vmax = compute_vmin_vmax(data)
            ax = axes[i]
            im = ax.imshow(data, vmin=vmin, vmax=vmax)
            ax.set_title(os.path.basename(filename))
            fig.colorbar(im, ax=ax)

        for j in range(num_images, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()


#### 1. Extract multilooked C2 matrix from NISAR GSLC data
##### using **`polsartools.import_nisar_gslc`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/sensors/nisar.html#gslc-nisar-gslc) or run `print(polsartools.import_nisar_gslc.__doc__)`


In [3]:
inFile = r"E:\\nisar_data\\NISAR_L2_PR_GSLC_025_146_A_172_2005_DHDH_A_20260718T090230_20260718T090305_P05023_N_F_J_001.h5"

# The following function extracts a multilooked C2 matrix from NISAR dual-polarimetric GSLC data.
# Provide path to the NISAR GSLC .h5 file.

pst.import_nisar_gslc(inFile, azlks=2, rglks=1,
                      mat='C2',  # matrix type C2/Sxy
                      fmt='tif',  # output format tif//bin, 
                      comp=False,  # Tiff compression (optional)
                      out_dir=None,  # path to out_dir (optional)
                      )  # 5m,10m SLC spacing

Error: ``E:\nisar_data\NISAR_L2_PR_GSLC_025_146_A_172_2005_DHDH_A_20260718T090230_20260718T090305_P05023_N_F_J_001.h5`` does not exist


RuntimeError: Invalid .h5 file or structure!!

#### 2. Speckle filtering using polarimetric refined-Lee filter (optional)

In [ ]:
c2Folder = os.path.join(inFile.split('.h5')[0],'C2HX')
window_size = 3
pst.filter_refined_lee(c2Folder,win=window_size,
                  fmt='tif', # output format tif//bin
                  sub_dir=True, # creates a sub directory with filtered data
                  cog=False, ovr=[2, 4, 8, 16], # COG prameters
                  comp=False, # Tiff compression (optional)
        )

#### 3. Generate a quick look RGB from C2 matrix (optional)

In [ ]:
c2Folder =  os.path.join(inFile.split('.h5')[0],f"rlee_{window_size}x{window_size}",'C2HX')

# pst.rgb_dp(c2Folder,type=1)
# pst.rgb_dp(c2Folder,type=2)
# pst.rgb_dp(c2Folder,type=3)
pst.rgb_dp(c2Folder,type=4)

#### 4. Generating some polarimetric parameters from C2 matrix

##### 4.1 Barakat Degree of polarization 
##### using `polsartools.dop_dp` function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/DOP_dp.html) or run `print(polsartools.dop_dp.__.doc__)`


In [ ]:

pst.dop_dp(c2Folder)


##### Preview the output

In [ ]:

plot_images(c2Folder+'/dopdp.tif', lower_percentile=2, upper_percentile=98)

##### 4.2. Entropy-Alpha (H/α) parameters 
##### using **`polsartools.h_alphadp`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/halphadxp.html) or run `print(polsartools.h_alphadp.__.doc__)`


In [ ]:
pst.h_alpha_dp(c2Folder)


##### Preview the output

In [ ]:


plot_images(c2Folder+'/Hdp.tif', 
            c2Folder + '/alphadp.tif',
            lower_percentile=2, upper_percentile=98)



##### Plot Entropy-Alpha plot for dual-pol
##### using **`polsartools.plot_h_alpha_dp`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/halpha_plot_dp.html) or run `print(polsartools.plot_h_alpha_dp.__.doc__)`

In [ ]:
hFile = c2Folder + '/Hdp.tif'
alphaFile = c2Folder + '/alphadp.tif'

pst.plot_h_alpha_dp(hFile,alphaFile,norm='log')

##### 4.3. Shannon Entropy parameters (SE is the sum of contributions from intensity (SEI) and polarimetry (SEP))
##### using **`polsartools.shannon_h_dp`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/shannon_h_dp.html) or run `print(polsartools.shannon_h_dp.__.doc__)`

In [ ]:
pst.shannon_h_dp(c2Folder,win = 3)

##### Preview the output

In [ ]:

plot_images(c2Folder+'/H_Shannon.tif', 
            c2Folder + '/HI_Shannon.tif',
            c2Folder + '/HP_Shannon.tif',
            
            lower_percentile=2, upper_percentile=98)


##### 4.4. vegetation indices, Radar vegetation index (RVI), Dual-pol Radar Vegetation Index (DpRVI)

##### using **`polsartools.rvi_dp`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/RVI_dp.html) or run `print(polsartools.rvi_dp.__.doc__)`

##### using **`polsartools.dprvi`** function, [click here for documentation](https://polsartools.readthedocs.io/en/latest/files/functions/dual_pol/DpRVI.html) or run `print(polsartools.dprvi.__.doc__)`




In [ ]:

pst.rvi_dp(c2Folder)

pst.dprvi(c2Folder)


##### Preview the output

In [ ]:

plot_images(c2Folder+'/rvidp.tif', 
            c2Folder + '/dprvi.tif',           
            lower_percentile=2, upper_percentile=98)
